# 05b — Uncertainty Calibration · Phase 5B

**Purpose:** Load Phase 5A predictions, apply temperature scaling, evaluate before/after calibration.

**Run AFTER:**
1. `python scripts/phase5a_save_predictions.py` — creates correct NPZ files
2. `python scripts/phase5b_calibrate.py --all` — runs calibration

**Research Gap addressed:** Gap 2 — Calibration

**Key question:** Are the GAT uncertainty intervals well-calibrated?
- 90% prediction interval should contain true value 90% of the time (PICP=0.90)
- Temperature scaling corrects over/underconfidence post-hoc
- No model retraining — only T is learned on validation set
---

# Setup

In [1]:
import os, sys, pickle
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from wildfire_gnn.utils.config import load_yaml_config
from wildfire_gnn.models.calibration import (
    TemperatureScaling,
    compute_picp, compute_mpiw,
    reliability_curve, compute_ace, compute_ence,
    compute_all_calibration_metrics,
)

config   = load_yaml_config(PROJECT_ROOT / 'configs' / 'gnn_config.yaml')
p        = config['paths']
PRED_DIR = PROJECT_ROOT / 'reports' / 'predictions'
TBL_DIR  = PROJECT_ROOT / 'reports' / 'tables'
FIG_DIR  = PROJECT_ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

ARCH = 'GAT'   # Change to 'GCN' or 'GraphSAGE' for ablation
print(f'Architecture : {ARCH}')
print(f'Predictions  : {PRED_DIR}')

Architecture : GAT
Predictions  : d:\wildfire\spatiotemporal_wildfire_gnn\reports\predictions


# Load Phase 5 A prediction

In [2]:
pred_path = PRED_DIR / f'phase5a_{ARCH.lower()}_preds.npz'

if not pred_path.exists():
    print(f'ERROR: {pred_path.name} not found.')
    print('Run: python scripts/phase5a_save_predictions.py')
else:
    data = np.load(pred_path)
    print(f'Loaded: {pred_path.name}')
    print(f'Keys: {list(data.files)}')
    print()

    y_true_bp   = data['y_true_bp']     # original scale
    y_pred_bp   = data['y_pred_bp']     # original scale
    mean_pred_t = data['mean_pred_t']   # transformed scale
    std_pred    = data['std_pred']      # epistemic uncertainty (transformed)
    aleatoric   = data['aleatoric']     # aleatoric uncertainty
    total_unc   = data['total_unc']     # combined uncertainty (transformed)
    samples     = data['samples']       # (30, N_test) MC passes

    # Verify R² matches confirmed result
    ss_res = np.sum((y_true_bp - y_pred_bp)**2)
    ss_tot = np.sum((y_true_bp - y_true_bp.mean())**2)
    r2_check = float(1 - ss_res/ss_tot)
    print(f'R² verification: {r2_check:.4f}  (should be ~0.766 for GAT)')
    print(f'Samples shape  : {samples.shape}  (30 MC passes × {len(y_true_bp):,} test nodes)')
    print(f'\nEpistemic std : mean={std_pred.mean():.4f}  max={std_pred.max():.4f}')
    print(f'Aleatoric std : mean={aleatoric.mean():.4f}  max={aleatoric.max():.4f}')
    print(f'Total unc     : mean={total_unc.mean():.4f}')

Loaded: phase5a_gat_preds.npz
Keys: ['y_true_bp', 'y_pred_bp', 'mean_pred_t', 'log_var_mean', 'y_true_t', 'std_pred', 'aleatoric', 'total_unc', 'samples', 'test_idx']

R² verification: 0.7651  (should be ~0.766 for GAT)
Samples shape  : (30, 57531)  (30 MC passes × 57,531 test nodes)

Epistemic std : mean=0.1384  max=3.6261
Aleatoric std : mean=0.8754  max=1.7775
Total unc     : mean=0.8908


# Convert uncertainty to original BP scale

In [3]:
# The uncertainty is in transformed (near-Gaussian) scale.
# We need to convert to original BP scale using the chain rule:
#   std_bp ≈ std_t × |d(inverse_transform)/d(t)|

trans_path = PROJECT_ROOT / p['target_transformer']
with open(trans_path, 'rb') as f:
    transformer = pickle.load(f)

eps   = 0.01
up    = transformer.inverse_transform((mean_pred_t + eps).reshape(-1,1)).ravel()
dn    = transformer.inverse_transform((mean_pred_t - eps).reshape(-1,1)).ravel()
deriv = np.abs((up - dn) / (2 * eps))   # local derivative

total_unc_bp = total_unc * deriv   # total uncertainty in BP scale
ep_bp        = std_pred  * deriv   # epistemic in BP scale
al_bp        = aleatoric * deriv   # aleatoric in BP scale

print(f'Total uncertainty (BP scale):')
print(f'  mean={total_unc_bp.mean():.5f}  max={total_unc_bp.max():.5f}')
print(f'Epistemic (BP scale):')
print(f'  mean={ep_bp.mean():.5f}')
print(f'Aleatoric (BP scale):')
print(f'  mean={al_bp.mean():.5f}')

Total uncertainty (BP scale):
  mean=0.02731  max=0.35056
Epistemic (BP scale):
  mean=0.00566
Aleatoric (BP scale):
  mean=0.02641


# Calibration before Temperature Scaling

In [4]:
print('='*60)
print(f'  {ARCH} — Calibration BEFORE Temperature Scaling')
print('='*60)

metrics_before = compute_all_calibration_metrics(
    y_true_bp, y_pred_bp, total_unc_bp,
    label=f'{ARCH} before', verbose=True
)

# Interpretation
picp90 = metrics_before['picp_90']
if picp90 < 0.85:
    interp = '⚠ OVERCONFIDENT — intervals too narrow, T will be > 1'
elif picp90 > 0.95:
    interp = '⚠ UNDERCONFIDENT — intervals too wide, T will be < 1'
else:
    interp = '✓ WELL-CALIBRATED — minor adjustment expected'
print(f'\n  Interpretation: {interp}')

  GAT — Calibration BEFORE Temperature Scaling

  Calibration metrics — GAT before
    PICP-50%  = 0.8554  (target 0.500, ✗)
    PICP-90%  = 0.9739  (target 0.900, ✗)
    PICP-95%  = 0.9841  (target 0.950, ✓)
    MPIW-90%  = 0.08984  (interval width)
    ACE       = +0.2353  (underconfident)
    ENCE      = 0.4109  (target < 0.10)

  Interpretation: ⚠ UNDERCONFIDENT — intervals too wide, T will be < 1


# Fit Temperature Scaling on Validation Set

In [ ]:
# Temperature is fit on VALIDATION SET ONLY — not test set
# Load validation predictions from model

from wildfire_gnn.models.gnn import build_model

CKPT_DIR    = PROJECT_ROOT / 'checkpoints'
ARCHIVE_DIR = CKPT_DIR / 'archive'
ckpt_name   = f'phase5a_{ARCH.lower()}_best.pt'
ckpt_path   = ARCHIVE_DIR / ckpt_name
if not ckpt_path.exists():
    ckpt_path = CKPT_DIR / f'gnn_{ARCH.lower()}_best.pt'

print(f'Loading checkpoint: {ckpt_path.name}')
ckpt  = torch.load(ckpt_path, map_location='cpu', weights_only=False)
m_cfg = ckpt.get('config', config)['model']
model = build_model(
    architecture = ARCH,
    in_channels  = m_cfg['in_channels'],
    hidden       = m_cfg['hidden_channels'],
    num_layers   = m_cfg.get('num_layers', 4),
    heads        = m_cfg.get('heads', 8),
    dropout      = m_cfg.get('dropout', 0.3),
)
model.load_state_dict(ckpt['model_state'])

# Load graph for val split
graph = torch.load(PROJECT_ROOT / p['graph_data'],
                   map_location='cpu', weights_only=False)

# Run 30 MC passes on validation nodes
print('Running 30 MC passes on validation split...')
model.train()  # dropout ON
val_means, val_lvs = [], []
with torch.no_grad():
    for _ in range(30):
        mean, lv = model(graph.x, graph.edge_index)
        val_means.append(mean[graph.val_mask].numpy())
        val_lvs.append(lv[graph.val_mask].numpy())

val_samples  = np.stack(val_means)          # (30, N_val)
val_mean_t   = val_samples.mean(axis=0)
val_std_t    = val_samples.std(axis=0)
val_al_t     = np.sqrt(np.exp(np.stack(val_lvs).mean(axis=0)))
val_total_t  = np.sqrt(val_std_t**2 + val_al_t**2)
val_y_true_t = graph.y[graph.val_mask].numpy().ravel()

print(f'Val nodes : {len(val_y_true_t):,}')
print(f'Val std   : mean={val_total_t.mean():.4f}')

# Fit temperature
print('\nFitting temperature...')
ts = TemperatureScaling()
ts.fit(val_mean_t, val_total_t, val_y_true_t)
print(f'\nTemperature T = {ts.T:.4f}')

Loading checkpoint: gnn_gat_best.pt
Running 30 MC passes on validation split...


# Apply Temperature Scaling and Evaluate after

In [ ]:
# Apply temperature to test predictions
total_unc_calibrated_bp = ts.scale(total_unc_bp)

print('='*60)
print(f'  {ARCH} — Calibration AFTER Temperature Scaling (T={ts.T:.4f})')
print('='*60)

metrics_after = compute_all_calibration_metrics(
    y_true_bp, y_pred_bp, total_unc_calibrated_bp,
    label=f'{ARCH} after', verbose=True
)

# Before vs After comparison
print('\n  Summary — Before vs After:')
print(f'  PICP-90% : {metrics_before["picp_90"]:.4f} → {metrics_after["picp_90"]:.4f}  '
      f'(target: 0.900)')
print(f'  ACE      : {metrics_before["ace"]:+.4f} → {metrics_after["ace"]:+.4f}  '
      f'(target: ~0.000)')
print(f'  ENCE     : {metrics_before["ence"]:.4f} → {metrics_after["ence"]:.4f}  '
      f'(target: < 0.10)')

# Reliability Diagram

In [ ]:
exp_b, act_b = metrics_before['expected'], metrics_before['actual']
exp_a, act_a = metrics_after['expected'],  metrics_after['actual']

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot([0,1],[0,1],'k--',lw=1.5, label='Perfect calibration')
ax.plot(exp_b, act_b, 'o-', color='#E74C3C', lw=2, ms=5,
        label=f'Before (T=1.0)  PICP-90={metrics_before["picp_90"]:.3f}')
ax.plot(exp_a, act_a, 's-', color='#2ECC71', lw=2, ms=5,
        label=f'After  (T={ts.T:.3f}) PICP-90={metrics_after["picp_90"]:.3f}')
ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_xlabel('Expected coverage', fontsize=12)
ax.set_ylabel('Actual coverage (PICP)', fontsize=12)
ax.set_title(f'Phase 5B — {ARCH} Reliability Diagram\nPerfect calibration = diagonal')
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / f'p5b_{ARCH.lower()}_reliability.png', dpi=150)
plt.show()
print('Figure saved')

# 90 % Prediction Interval Coverage Plot

In [ ]:
rng      = np.random.default_rng(42)
idx      = rng.choice(len(y_true_bp), 2000, replace=False)
sort_idx = idx[np.argsort(y_true_bp[idx])]
z        = 1.645   # 90% interval
yt       = y_true_bp[sort_idx]
mp       = y_pred_bp[sort_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, std, label, color in zip(
    axes,
    [total_unc_bp[sort_idx], total_unc_calibrated_bp[sort_idx]],
    ['Before scaling', f'After scaling (T={ts.T:.3f})'],
    ['#E74C3C', '#2ECC71']
):
    lo   = mp - z * std
    hi   = mp + z * std
    picp = float(np.mean((yt >= lo) & (yt <= hi)))
    x    = np.arange(len(yt))
    ax.fill_between(x, lo, hi, alpha=0.3, color=color,
                    label=f'90% PI (PICP={picp:.3f})')
    ax.plot(x, yt, 'k.', ms=1.5, alpha=0.4, label='True BP')
    ax.plot(x, mp, color=color, lw=1, alpha=0.8, label='Pred mean')
    ax.set_title(f'{ARCH} — 90% Intervals\n{label}')
    ax.set_xlabel('Nodes (sorted by true BP)')
    ax.set_ylabel('Burn Probability')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIG_DIR / f'p5b_{ARCH.lower()}_interval_coverage.png', dpi=150)
plt.show()
print('Figure saved')

# Save Calibration Results

In [ ]:
import pandas as pd

cal_df = pd.DataFrame([
    {'arch': ARCH, 'stage': 'before', 'T': 1.0,
     'picp_50': metrics_before['picp_50'],
     'picp_90': metrics_before['picp_90'],
     'picp_95': metrics_before['picp_95'],
     'mpiw_90': metrics_before['mpiw_90'],
     'ace':     metrics_before['ace'],
     'ence':    metrics_before['ence']},
    {'arch': ARCH, 'stage': 'after', 'T': ts.T,
     'picp_50': metrics_after['picp_50'],
     'picp_90': metrics_after['picp_90'],
     'picp_95': metrics_after['picp_95'],
     'mpiw_90': metrics_after['mpiw_90'],
     'ace':     metrics_after['ace'],
     'ence':    metrics_after['ence']},
])

out = TBL_DIR / f'phase5b_{ARCH.lower()}_calibration.csv'
cal_df.to_csv(out, index=False)
print(f'Saved: {out.name}')
print(cal_df.to_string(index=False))

# Phase 5 B Completion checklist

In [ ]:
print('='*55)
print('  PHASE 5B COMPLETION CHECKLIST')
print('='*55)

items = [
    ('NPZ predictions loaded',          True),
    ('R² verified from NPZ',            r2_check > 0.70),
    ('Temperature T fitted on val',     ts.fitted),
    ('T in reasonable range',           0.01 < ts.T < 10.0),
    ('PICP-90 after in [0.85,0.95]',   0.85 < metrics_after['picp_90'] < 0.95),
    ('ACE improved after scaling',      abs(metrics_after['ace']) <= abs(metrics_before['ace'])),
    ('Reliability figure saved',        (FIG_DIR / f'p5b_{ARCH.lower()}_reliability.png').exists()),
    ('Interval figure saved',           (FIG_DIR / f'p5b_{ARCH.lower()}_interval_coverage.png').exists()),
    ('Calibration CSV saved',           (TBL_DIR / f'phase5b_{ARCH.lower()}_calibration.csv').exists()),
]

all_ok = True
for label, ok in items:
    sym = '✓' if ok else '✗'
    print(f'  {sym}  {label}')
    all_ok = all_ok and ok

print('='*55)
if all_ok:
    print('  ALL CHECKS PASSED — proceed to Phase 5D (Intervention)')
else:
    print('  SOME CHECKS FAILED — review above')